In [1]:
from pyflink.table import (
    EnvironmentSettings,
    TableEnvironment
)

In [2]:
env_settings = (
    EnvironmentSettings.new_instance()
    .in_streaming_mode()
    .build()
)
t_env = TableEnvironment.create(env_settings)
conf = t_env.get_config().get_configuration()

/usr/local/lib/python3.11/dist-packages/apache_beam/runners/portability/stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
conf.set_string("execution.target", "remote")
conf.set_string("rest.address", "jobmanager")
conf.set_string("rest.port", "8081")
conf.set_string("parallelism.default", "1")

In [4]:
t_env.execute_sql("""
CREATE TABLE source_table (
    id INT,
    name STRING,
    age INT
) WITH (
    'connector' = 'datagen',
    'rows-per-second' = '2'
)
""")

In [5]:
t_env.execute_sql("""
CREATE TABLE sink_table (
    id INT,
    name STRING,
    age INT
) WITH (
    'connector' = 'filesystem',
    'path' = 'file:///workspace/output/json_sink',
    'format' = 'json'
)
""")

In [6]:
result = t_env.execute_sql("""
INSERT INTO sink_table
SELECT id, name, age FROM source_table
""")

In [13]:
result.get_job_client().cancel()

In [7]:
t_env.execute_sql("""
INSERT INTO sink_table
SELECT * FROM (VALUES (1, 'ABC', 24), (2, 'DEF', 25)) AS t(id, name, age)
""")

In [8]:
source_table = t_env.from_elements(
    [(1, 'ABC', 26), (2, 'DEF', 27)],
    ['id', 'name', 'age']
)
source_table.print_schema()

(
  `id` BIGINT,
  `name` STRING,
  `age` BIGINT
)


In [9]:
t_env.create_temporary_view("source_view", source_table)

t_env.execute_sql("""
INSERT INTO sink_table
SELECT CAST(id AS INT), name, CAST(age AS INT) FROM source_view
""")

In [11]:
import os
import csv
import pandas as pd
from datetime import datetime
from pyflink.table.expressions import col, lit
from pyflink.table import DataTypes
from pyflink.table.window import Slide, Tumble
from pyflink.table.udf import udf

In [45]:
rows = []
with open("ADAUSDT-aggTrades-2025-09-27.csv") as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader, 1):
        rows.append([
            int(row[0]),
            float(row[1]),
            float(row[2]),
            int(row[3]),
            int(row[4]),
            int(row[5]),
            row[6].strip().lower() == "true",
            row[7].strip().lower() == "true"
        ])
        if i % 1000 == 0:
            break

In [46]:
print(rows[-1], len(rows))

[411115766, 0.7911, 110.3, 711709969, 711709969, 1758933771529397, False, True] 1000


In [47]:
t_env \
    .from_elements(rows, ["agg_trade_id","price","quantity","first_trade_id","last_trade_id","timestamp","is_buyer_maker","is_best_match"]) \
    .print_schema()

(
  `agg_trade_id` BIGINT,
  `price` DOUBLE,
  `quantity` DOUBLE,
  `first_trade_id` BIGINT,
  `last_trade_id` BIGINT,
  `timestamp` BIGINT,
  `is_buyer_maker` BOOLEAN,
  `is_best_match` BOOLEAN
)


In [48]:
schema = DataTypes.ROW([
  DataTypes.FIELD("agg_trade_id", DataTypes.BIGINT()),
  DataTypes.FIELD("price", DataTypes.DOUBLE()),
  DataTypes.FIELD("quantity", DataTypes.DOUBLE()),
  DataTypes.FIELD("first_trade_id", DataTypes.BIGINT()),
  DataTypes.FIELD("last_trade_id", DataTypes.BIGINT()),
  DataTypes.FIELD("timestamp", DataTypes.BIGINT()),
  DataTypes.FIELD("is_buyer_maker", DataTypes.BOOLEAN()),
  DataTypes.FIELD("is_best_match", DataTypes.BOOLEAN())
])

In [49]:
source_table = t_env \
    .from_elements(rows, schema)

In [50]:
source_table.print_schema()

(
  `agg_trade_id` BIGINT,
  `price` DOUBLE,
  `quantity` DOUBLE,
  `first_trade_id` BIGINT,
  `last_trade_id` BIGINT,
  `timestamp` BIGINT,
  `is_buyer_maker` BOOLEAN,
  `is_best_match` BOOLEAN
)


In [53]:
source_table.limit(10).execute().print()

+----+----------------------+--------------------------------+--------------------------------+----------------------+----------------------+----------------------+----------------+---------------+
| op |         agg_trade_id |                          price |                       quantity |       first_trade_id |        last_trade_id |            timestamp | is_buyer_maker | is_best_match |
+----+----------------------+--------------------------------+--------------------------------+----------------------+----------------------+----------------------+----------------+---------------+
| +I |            411114767 |                         0.7918 |                          111.5 |            711706932 |            711706932 |     1758931200686229 |           TRUE |          TRUE |
| +I |            411114768 |                         0.7918 |                          252.5 |            711706933 |            711706933 |     1758931200711075 |           TRUE |          TRUE |
| +I |    

In [54]:
df = source_table.to_pandas()

In [55]:
df.head()

,agg_trade_id,price,quantity,first_trade_id,last_trade_id,timestamp,is_buyer_maker,is_best_match
0,411114767,0.7918,111.5,711706932,711706932,1758931200686229,True,True
1,411114768,0.7918,252.5,711706933,711706933,1758931200711075,True,True
2,411114769,0.7918,240.6,711706934,711706937,1758931202600822,True,True
3,411114770,0.7917,43.1,711706938,711706940,1758931204869784,True,True
4,411114771,0.7917,22.6,711706941,711706941,1758931204879746,True,True


In [56]:
t_env.execute_sql("SELECT * FROM source_table LIMIT 10").print()

+----+-------------+--------------------------------+-------------+
| op |          id |                           name |         age |
+----+-------------+--------------------------------+-------------+
| +I |  -680534815 | ef5909c42b695492b8d6528b5d5... |  1328139289 |
| +I | -1540431703 | ce28372d3a9edb38fc88ed28259... | -1968108023 |
| +I |    37799567 | 4957d59d091f592510c5b4e6fb1... |  -788593778 |
| +I |  -706151961 | fd95285cc718c4121cb7a9fd297... |   952060750 |
| +I |  -883903081 | 24ef6bd4376a8e3138dead99732... |   544732273 |
| +I |  -506114951 | dd2489c5b8427cfc43fa2d34888... |  -904441876 |
| +I |  1740924011 | 5ae2c1b91fe996dae0e14270c17... |  -228110987 |
| +I |  -997376322 | c9f5496182c99d3bc2fca34a9a6... |  -564155950 |
| +I |  1231404089 | 12285ae5e97af25198260efe057... |   330036044 |
| +I | -1999856426 | e12fc4e82928383adc734cfb96a... |  -609079532 |
+----+-------------+--------------------------------+-------------+
10 rows in set


In [71]:
t_env.execute_sql("DROP TABLE IF EXISTS source_table")
t_env.execute_sql("""
CREATE table source_table (
    agg_trade_id STRING,
    price STRING,
    quantity STRING,
    first_trade_id STRING,
    last_trade_id STRING,
    `timestamp` STRING,
    is_buyer_maker STRING,
    is_best_match STRING
) WITH (
    'connector' = 'filesystem',
    'path' = 'file:///workspace/ADAUSDT-aggTrades-2025-09-27.csv',
    'format' = 'csv'
)
""")

# ✅ Cast after loading
t_env.execute_sql("""
SELECT
    CAST(agg_trade_id AS BIGINT) AS agg_trade_id,
    CAST(price AS DOUBLE) AS price,
    CAST(quantity AS DOUBLE) AS quantity,
    CAST(first_trade_id AS BIGINT) AS first_trade_id,
    CAST(last_trade_id AS BIGINT) AS last_trade_id,
    CAST(`timestamp` AS BIGINT) AS ts,
    CAST(is_buyer_maker AS BOOLEAN) AS is_buyer_maker,
    CAST(is_best_match AS BOOLEAN) AS is_best_match
FROM source_table
LIMIT 10
""").print()

+----+----------------------+--------------------------------+--------------------------------+----------------------+----------------------+----------------------+----------------+---------------+
| op |         agg_trade_id |                          price |                       quantity |       first_trade_id |        last_trade_id |                   ts | is_buyer_maker | is_best_match |
+----+----------------------+--------------------------------+--------------------------------+----------------------+----------------------+----------------------+----------------+---------------+
| +I |            411114767 |                         0.7918 |                          111.5 |            711706932 |            711706932 |     1758931200686229 |           TRUE |          TRUE |
| +I |            411114768 |                         0.7918 |                          252.5 |            711706933 |            711706933 |     1758931200711075 |           TRUE |          TRUE |
| +I |    

In [73]:
source_table = t_env.from_path("source_table")

In [74]:
source_table.print_schema()

(
  `agg_trade_id` STRING,
  `price` STRING,
  `quantity` STRING,
  `first_trade_id` STRING,
  `last_trade_id` STRING,
  `timestamp` STRING,
  `is_buyer_maker` STRING,
  `is_best_match` STRING
)


In [75]:
t_env.execute_sql("DROP TABLE IF EXISTS source_table")
t_env.execute_sql("""
CREATE table source_table (
    agg_trade_id BIGINT,
    price DOUBLE,
    quantity DOUBLE,
    first_trade_id BIGINT,
    last_trade_id BIGINT,
    `timestamp` BIGINT,
    is_buyer_maker BOOLEAN,
    is_best_match BOOLEAN
) WITH (
    'connector' = 'filesystem',
    'path' = 'file:///workspace/ADAUSDT-aggTrades-2025-09-27.csv',
    'format' = 'csv'
)
""")

# ✅ Cast after loading
t_env.execute_sql("""
SELECT * FROM source_table LIMIT 10
""").print()

+----+----------------------+--------------------------------+--------------------------------+----------------------+----------------------+----------------------+----------------+---------------+
| op |         agg_trade_id |                          price |                       quantity |       first_trade_id |        last_trade_id |            timestamp | is_buyer_maker | is_best_match |
+----+----------------------+--------------------------------+--------------------------------+----------------------+----------------------+----------------------+----------------+---------------+
| +I |            411114767 |                         0.7918 |                          111.5 |            711706932 |            711706932 |     1758931200686229 |           TRUE |          TRUE |
| +I |            411114768 |                         0.7918 |                          252.5 |            711706933 |            711706933 |     1758931200711075 |           TRUE |          TRUE |
| +I |    

In [79]:
t_env.from_path("source_table").print_schema()

(
  `agg_trade_id` BIGINT,
  `price` DOUBLE,
  `quantity` DOUBLE,
  `first_trade_id` BIGINT,
  `last_trade_id` BIGINT,
  `timestamp` BIGINT,
  `is_buyer_maker` BOOLEAN,
  `is_best_match` BOOLEAN
)


In [82]:
t_env.from_path("source_table").to_pandas().head(10)

,agg_trade_id,price,quantity,first_trade_id,last_trade_id,timestamp,is_buyer_maker,is_best_match
0,411114767,0.7918,111.5,711706932,711706932,1758931200686229,True,True
1,411114768,0.7918,252.5,711706933,711706933,1758931200711075,True,True
2,411114769,0.7918,240.6,711706934,711706937,1758931202600822,True,True
3,411114770,0.7917,43.1,711706938,711706940,1758931204869784,True,True
4,411114771,0.7917,22.6,711706941,711706941,1758931204879746,True,True
5,411114772,0.7917,669.9,711706942,711706945,1758931205396640,True,True
6,411114773,0.7916,81.2,711706946,711706949,1758931205433486,True,True
7,411114774,0.7917,70.1,711706950,711706951,1758931209806960,False,True
8,411114775,0.7917,624.0,711706952,711706954,1758931211082402,False,True
9,411114776,0.7917,31.6,711706955,711706955,1758931211824816,False,True


In [ ]:

t_env.execute_sql("""
CREATE TEMPORARY VIEW source_view AS
SELECT * FROM some_table
""")

In [88]:
t_env.execute_sql("DROP TEMPORARY VIEW IF EXISTS source_view")
t_env.create_temporary_view("source_view", t_env.from_path("source_table"))

In [89]:
print(t_env.list_temporary_views())

['UnnamedTable$0', 'UnnamedTable$1', 'source_view']


In [90]:
print(t_env.list_temporary_tables())

['UnnamedTable$0', 'UnnamedTable$1', 'source_view']


In [91]:
t_env.execute_sql("select * from source_view limit 10").print()

+----+----------------------+--------------------------------+--------------------------------+----------------------+----------------------+----------------------+----------------+---------------+
| op |         agg_trade_id |                          price |                       quantity |       first_trade_id |        last_trade_id |            timestamp | is_buyer_maker | is_best_match |
+----+----------------------+--------------------------------+--------------------------------+----------------------+----------------------+----------------------+----------------+---------------+
| +I |            411114767 |                         0.7918 |                          111.5 |            711706932 |            711706932 |     1758931200686229 |           TRUE |          TRUE |
| +I |            411114768 |                         0.7918 |                          252.5 |            711706933 |            711706933 |     1758931200711075 |           TRUE |          TRUE |
| +I |    